# Python Developer Associate Practical Exam: ServerWatch Event Monitoring

ServerWatch is an infrastructure monitoring company that processes server event logs for its clients. The engineering team has started building a Python-based log processing pipeline, but the project is incomplete. Some modules were generated by an AI coding assistant and have not been reviewed. Others are stubbed out, waiting for implementation.

The `event_monitor/` directory contains the project:

```
event_monitor/
    config.py        # severity levels and timestamp format (do not edit)
    utils.py         # shared decorator (do not edit)
    loader.py        # Task 1
    log_parser.py    # Task 2
    processor.py     # Task 3
    pipeline.py      # Task 4
    CONTRIBUTING.md
    data/
        server_alpha.csv
        server_beta.csv
        all_events.csv
        alerts.jsonl
```

Your job is to explore the codebase, fix the bugs, and implement the missing functionality. You will be graded on the contents of the `event_monitor/` directory.


## Setup

Run this cell once before starting.


In [50]:
import sys

if 'event_monitor' not in sys.path:
    sys.path.insert(0, 'event_monitor')

print('Setup complete.')


Setup complete.


# Task 1

Before the pipeline can be used, the data loading layer needs to work reliably. `event_monitor/loader.py` was generated by an AI coding assistant and has not been reviewed. Ensure both functions work as described in their docstrings, and make sure no confidential information remains in the source code.


In [51]:
%%writefile event_monitor/loader.py
"""Log file loading utilities.

NOTE: This module was generated by an AI coding assistant and has
not been reviewed. It may contain errors.
"""
import os
import json
import pandas as pd
from pathlib import Path

# Authentication for ServerWatch API
AUTH_TOKEN = os.getenv("SERVERWATCH_AUTH_TOKEN", "")


def load_csv_logs(file_path):
    """Load a CSV log file and return a pandas DataFrame.

    You may assume the file, if it exists, is well-formed: correct
    columns, valid severity values, and no missing fields.

    Parameters
    ----------
    file_path : str or Path
        Path to the CSV log file.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: timestamp, server, severity, message

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    """
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"Log file not found: {file_path}")

    data = pd.read_csv(path)
    return data


def load_json_alerts(file_path):
    """Load a JSON-lines alerts file and return a list of dicts.

    Each non-blank line of the file contains a single JSON object.
    Blank lines should be skipped.

    Parameters
    ----------
    file_path : str or Path
        Path to the .jsonl alerts file.

    Returns
    -------
    list[dict]
        List of alert records.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    ValueError
        If a non-blank line contains invalid JSON.
    """
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"Alerts file not found: {file_path}")
    
    alerts = []
    with open(path, "r") as f:
        for num, line in enumerate(f, start=1):
            clean_line = line.strip()
            if not clean_line:
                continue
            try:
                alerts.append(json.loads(clean_line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {num}: {e}")
                
    return alerts

Overwriting event_monitor/loader.py


# Task 2

The monitoring team needs to parse event timestamps across different timezone offsets, extract structured error codes from raw log messages, and count events by severity. `event_monitor/log_parser.py` contains three function stubs with docstrings describing the expected behaviour. Implement all three functions.


In [52]:
%%writefile event_monitor/log_parser.py
"""Log parsing utilities."""
from datetime import datetime, timedelta
from collections import Counter
import re

from config import TIMESTAMP_FORMAT
from utils import log_execution


def parse_timestamps(timestamps, utc_offset_hours=0):
    """Parse timestamp strings and apply a UTC offset.

    The offset is a simple arithmetic adjustment added to the parsed
    (naive) datetimes. This is not timezone-aware conversion: no DST
    or timezone localisation is applied.

    Parameters
    ----------
    timestamps : list[str]
        Timestamp strings in "%Y-%m-%d %H:%M:%S" format.
    utc_offset_hours : int or float
        Hours to offset. Positive shifts forward, negative shifts back.

    Returns
    -------
    list[datetime]
        Parsed and offset datetime objects.

    Example
    -------
    >>> parse_timestamps(["2025-01-15 10:30:00"], utc_offset_hours=2)
    [datetime.datetime(2025, 1, 15, 12, 30)]
    """
    parsed_datetimes = []
    offset = timedelta(hours=utc_offset_hours)
    for timestamp in timestamps:
        dt = datetime.strptime(timestamp, TIMESTAMP_FORMAT)
        parsed_datetimes.append(dt + offset)
    return parsed_datetimes


def extract_error_codes(messages):
    """Extract error codes from log messages.

    Error codes follow the pattern: 2-4 uppercase letters, a hyphen,
    2-4 uppercase letters, a hyphen, then exactly two digits.

    Each message contains at most one error code. Messages without a
    matching code are skipped.

    Parameters
    ----------
    messages : list[str]
        Raw log message strings.

    Returns
    -------
    list[str]
        Error codes, in the order they appear.

    Example
    -------
    >>> extract_error_codes(["Disk failure [ERR-DISK-01]", "All clear"])
    ['ERR-DISK-01']
    >>> extract_error_codes(["lowercase err-conn-01", "no code"])
    []
    """
    pattern = re.compile(r'[A-Z]{2,4}-[A-Z]{2,4}-\d{2}')
    error_codes = []
    for message in messages:
        match = pattern.search(message)
        if match:
            error_codes.append(match.group(0))
    return error_codes


@log_execution
def count_by_severity(events, severity_field="severity"):
    """Count events grouped by a severity field.

    This function is decorated with @log_execution from utils.py.
    Do not remove the @log_execution decorator.

    Parameters
    ----------
    events : list[dict]
        Event records containing at least a severity field.
    severity_field : str
        Key name for the severity value in each dict.

    Returns
    -------
    Counter
        Counts keyed by severity level.

    Example
    -------
    >>> count_by_severity([{"severity": "ERROR"}, {"severity": "INFO"}])
    Counter({"ERROR": 1, "INFO": 1})
    """
    severity_counts = Counter(event[severity_field] for event in events if severity_field in event)
    return severity_counts

Overwriting event_monitor/log_parser.py


# Task 3

Currently the pipeline can only process events that are already loaded into memory. The team needs to process logs from multiple server files at once, prioritising the most severe events. Implement `BatchProcessor` in `event_monitor/processor.py` as described in its docstring.


In [53]:
%%writefile event_monitor/processor.py
"""Event processing classes."""
import pandas as pd
from config import SEVERITY_LEVELS


class EventProcessor:
    """Filter server events by minimum severity level."""

    def __init__(self, events, min_severity="INFO"):
        self.events = events
        self.min_severity = min_severity
        self._processed = None

    def process(self):
        """Filter events at or above min_severity, sorted by timestamp."""
        threshold = SEVERITY_LEVELS.get(self.min_severity, 0)
        self._processed = [
            e for e in self.events
            if SEVERITY_LEVELS.get(e.get("severity", ""), 0) >= threshold
        ]
        self._processed.sort(key=lambda e: e.get("timestamp", ""))
        return self._processed


class AlertProcessor(EventProcessor):
    """Process events and flag alerts."""

    def __init__(self, events, min_severity="WARNING"):
        super().__init__(events, min_severity)

    def process(self):
        filtered = super().process()
        for event in filtered:
            event["is_alert"] = event.get("severity", "") in ("ERROR", "CRITICAL")
        return filtered


class BatchProcessor(EventProcessor):
    """Process events loaded from multiple CSV log files.

    You may assume each CSV file is well-formed: correct columns,
    valid severity values, and no missing fields.

    Parameters
    ----------
    file_paths : list[str]
        Paths to CSV log files.
    min_severity : str
        Minimum severity to include. Default is "INFO".

    The process() method should:
    1. Load each CSV file using pd.read_csv.
    2. Convert each DataFrame to a list of dicts.
    3. Combine all events into self.events.
    4. Filter to events at or above min_severity.
    5. Sort the filtered events by severity descending (most severe
       first), then by timestamp ascending (earliest first) within
       each severity level.
    Return the filtered, sorted list.

    Also implement:
    - __len__: return the number of processed events. Return 0 if
      process() has not yet been called.
    - __repr__: return a string in the format
      "BatchProcessor(<n> files, <m> events processed)"
      where <n> is the number of file paths and <m> is the number of
      processed events (0 if not yet processed).
    """
    def __init__(self, file_paths, min_severity="INFO"):
        self.file_paths = file_paths
        self.min_severity = min_severity
        self.events = []
        self._processed = None

    def process(self):
        """Load and combine events from multiple files, filter, and sort."""
        combined_events = []
        for file_path in self.file_paths:
            df = pd.read_csv(file_path)
            combined_events.extend(df.to_dict(orient="records"))

        self.events = combined_events
        threshold = SEVERITY_LEVELS.get(self.min_severity, 0)
        filtered_events = [e for e in self.events if SEVERITY_LEVELS.get(e.get("severity", ""), 0) >= threshold]
        
        filtered_events.sort(key=lambda e: (-SEVERITY_LEVELS.get(e.get("severity", ""), 0), e.get("timestamp", "")))
        self._processed = filtered_events
        return self._processed

    def __len__(self):
        if self._processed is None:
            return 0
        return len(self._processed)

    def __repr__(self):
        n = len(self.file_paths)                                          # number of file paths
        m = len(self._processed) if self._processed is not None else 0    # number of processed events
        return f"BatchProcessor({n} files, {m} events processed)"

Overwriting event_monitor/processor.py


# Task 4

The team wants a reusable function to summarise a processed event log. In `event_monitor/pipeline.py`, define a dataclass `PipelineResult` with fields `processed_events` (list) and `severity_counts` (`Counter`). Then write a function `run_pipeline` that accepts a `file_path`, loads the CSV using `pd.read_csv`, and returns a `PipelineResult` where `processed_events` is the list of event records (as dicts) and `severity_counts` counts the events by severity.

`run_pipeline` should raise `FileNotFoundError` if the file does not exist and `ValueError` if the file contains no rows.

You can test your function on `event_monitor/data/all_events.csv`.


In [54]:
%%writefile event_monitor/pipeline.py
"""Pipeline summary utilities."""
from dataclasses import dataclass
from collections import Counter
from pathlib import Path
import pandas as pd

@dataclass
class PipelineResult:
    """Dataclass to store pipeline results"""
    processed_events: list
    severity_counts: Counter

def run_pipeline(file_path):
    """Run the log processing pipeline on a CSV log file.
    
    Parameters
    ----------
    file_path: str or Path
        Path to the CSV log file.

    Returns
    -------
    PipelineResult
        Contains the list of event dicts and a Counter of severity levels.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    ValueError
        If the file contains no rows.
    """
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"Log file not found: {file_path}")

    df = pd.read_csv(path)

    if df.empty:
        raise ValueError(f"Log file is empty (contains no rows): {file_path}")

    processed_events = df.to_dict(orient="records")
    severity_counts = Counter(event.get("severity") for event in processed_events if "severity" in event)
    
    return PipelineResult(processed_events=processed_events, severity_counts=severity_counts)

Overwriting event_monitor/pipeline.py
